# Exercise 7 — Hardware validation with Jupyter (PYNQ-Z2)

This notebook is a **standalone hardware test bench** for the 4-bit parallel and shift registers. It interacts with the **real programmable logic** via `pynq.MMIO`; no `register_control.py` is needed.

**Deployment model:** program the *matching* `registers.bit` into the PL using **Vivado Hardware Manager first**. The notebook does **not** program or replace the bitstream and does not call `Overlay(...)`.

**Prerequisites**
- Run this notebook on the PYNQ-Z2 with its PYNQ Linux image, not inside the host devcontainer.
- The programmed design must include the Zynq PS, a **running 100 MHz FCLK_CLK0**, initialized AXI interconnect, and the `register_gpio` dual-channel AXI GPIO attached to `register_axi_wrapper`. Programming a `.bit` with Vivado **does not by itself guarantee PS clock/reset initialization**; if MMIO hangs or fails, check PS initialization and the currently running Linux configuration.
- Obtain the **AXI GPIO base address and address range from the exact Vivado Address Editor / exported hardware handoff** of the programmed design. Do not guess addresses or rely on a prior overlay's PL metadata.
- Only **one writer** may access these commands. There is no hardware ACK; eight uncached MMIO reads provide a settling interval, **not** completion confirmation.

**GPIO contract** (as in `docs/integration_pynq.md`): channel 1 = 9 output command bits; channel 2 = 4 input `q` bits. Bit mapping: `d[3:0]` = 0–3, `serial_in` = 4, `mode` = 5, `load_request` = 6, `shift_request` = 7, `reset_request` = 8.

**Safety:** stop if the bitstream/AXI address does not match. Accessing an invalid MMIO address may stall the kernel or cause a bus error.


## 1. Configuration — use the address of your programmed design

In [ ]:
from pynq import MMIO

# REQUIRED: copy these values from Vivado > Address Editor for `register_gpio`.
# Example format only: AXI_GPIO_BASE = 0x41200000 (NOT a verified address).
AXI_GPIO_BASE = None
AXI_GPIO_RANGE = 0x1000  # verify the segment size in Vivado too

assert isinstance(AXI_GPIO_BASE, int) and AXI_GPIO_BASE >= 0, (
    "Set AXI_GPIO_BASE to the `register_gpio` base address from the current Vivado design."
)
assert isinstance(AXI_GPIO_RANGE, int) and AXI_GPIO_RANGE >= 0x10
assert AXI_GPIO_BASE % 4 == 0, "Base address must be 32-bit aligned"
print(f"Using AXI GPIO at 0x{AXI_GPIO_BASE:08X}, range 0x{AXI_GPIO_RANGE:X}")

## 2. MMIO registers and command driver

AXI GPIO standard offsets: channel 1 data `0x00`, channel 1 direction `0x04`, channel 2 data `0x08`, channel 2 direction `0x0C`. Direction register: bit `0` = output, bit `1` = input.

The Python class below is intentionally **defined in this notebook**. It preserves the original test protocol, with no external `.py` dependency. The GPIO output register is not used as a reliable command readback; `shadow` tracks the last command word.


In [ ]:
# AXI GPIO register offsets
GPIO1_DATA = 0x00
GPIO1_TRI = 0x04
GPIO2_DATA = 0x08
GPIO2_TRI = 0x0C

DATA_MASK = 0x00F
SERIAL = 1 << 4
MODE = 1 << 5
LOAD = 1 << 6
SHIFT = 1 << 7
RESET = 1 << 8
COMMAND_MASK = 0x1FF
Q_MASK = 0xF

class RegisterControl:
    """Single-writer, non-acknowledged command protocol for the GPIO adapter."""
    def __init__(self, mmio):
        self.gpio = mmio
        self.shadow = 0
        # All channel-1 command bits output, all channel-2 q bits input.
        self.gpio.write(GPIO1_TRI, 0x00000000)
        self.gpio.write(GPIO2_TRI, 0xFFFFFFFF)
        self.write_command(0)
        self.reset()

    def _settle(self):
        """Uncached read transactions allow PL cycles; they are NOT an ACK."""
        value = 0
        for _ in range(8):
            value = self.gpio.read(GPIO2_DATA) & Q_MASK
        return value

    def write_command(self, value):
        value = int(value)
        if not 0 <= value <= COMMAND_MASK:
            raise ValueError("command must fit in 9 bits")
        self.gpio.write(GPIO1_DATA, value)
        self.shadow = value
        return self._settle()

    def reset(self):
        base = self.shadow & ~(LOAD | SHIFT | RESET)
        self.write_command(base | RESET)
        self.write_command(base)

    def operate(self, *, data=None, serial=None, load=False, shift=False):
        base = self.shadow & ~(LOAD | SHIFT | RESET)
        if data is not None:
            if not isinstance(data, int) or not 0 <= data <= 15:
                raise ValueError("data must be an integer from 0 to 15")
            base = (base & ~DATA_MASK) | data
        if serial is not None:
            if serial not in (0, 1):
                raise ValueError("serial must be 0 or 1")
            base = (base & ~SERIAL) | (int(serial) << 4)
        self.write_command(base)  # requests low: stable data and rearm
        self.write_command(base | (LOAD if load else 0) | (SHIFT if shift else 0))
        return self.write_command(base)

    def load(self, value):
        return self.operate(data=value, load=True)

    def shift_bit(self, bit):
        return self.operate(serial=bit, shift=True)

    def select(self, mode):
        if mode not in (0, 1):
            raise ValueError("mode must be 0 or 1")
        return self.write_command((self.shadow & ~MODE) | (int(mode) << 5))

    def states(self):
        saved_mode = int(bool(self.shadow & MODE))
        parallel_value = self.select(0)
        shift_value = self.select(1)
        self.select(saved_mode)
        return parallel_value, shift_value

    def read_selected(self):
        return self.gpio.read(GPIO2_DATA) & Q_MASK

print("Inline MMIO driver defined.")

## 3. Connect to the configured AXI GPIO

**Run only after checking the base address and PS/PL initialization.** An incorrect address may block MMIO accesses. The first reset below is a *functional request* (GPIO bit 8), **not** a reprogramming or infrastructure reset.

In [ ]:
gpio = MMIO(AXI_GPIO_BASE, AXI_GPIO_RANGE)
control = RegisterControl(gpio)
observed = control.states()
assert observed == (0, 0), f"Reset/readback failed: {observed}"
print("AXI GPIO connected; functional reset and both readbacks: OK")

## 4. Parallel register — all 16 values and output selection

In [ ]:
control.reset()
for value in range(16):
    control.load(value)
    observed = control.states()
    assert observed == (value, 0), (
        f"Parallel load {value:04b}: expected {(value, 0)}, got {observed}"
    )
assert control.select(0) == 15
assert control.select(1) == 0
assert control.select(0) == 15
print("Parallel load (0..15) and output selection: PASS")

## 5. SIPO register — insertion, shifting, overflow

In [ ]:
control.reset()
control.load(10)
for bit, expected in [(1, 1), (0, 2), (0, 4), (0, 8), (0, 0)]:
    control.shift_bit(bit)
    observed = control.states()
    assert observed == (10, expected), (
        f"Shift bit={bit}: expected {(10, expected)}, got {observed}"
    )
for bit, expected in [(1, 1), (0, 2), (1, 5), (1, 11)]:
    control.shift_bit(bit)
    observed = control.states()
    assert observed == (10, expected), (
        f"Shift bit={bit}: expected {(10, expected)}, got {observed}"
    )
print("SIPO shift sequences and parallel-state independence: PASS")

## 6. Requests held high — no repeated operation, plus retention

In [ ]:
control.reset()
base = 10
control.write_command(base)
control.write_command(base | LOAD)
assert control.states() == (10, 0)
control.write_command(3 | LOAD)  # change data while LOAD stays high
for _ in range(16):
    assert control.states() == (10, 0), "Held LOAD incorrectly re-triggered"
control.write_command(3)  # rearm

base = 3 | SERIAL
control.write_command(base)
control.write_command(base | SHIFT)
for _ in range(16):
    assert control.states() == (10, 1), "Held SHIFT incorrectly re-triggered"
control.write_command(base)
for value in (0, 15, 6):
    control.write_command(value)
    assert control.states() == (10, 1), "Unexpected state change with requests low"
print("Held requests and retention: PASS")

## 7. Simultaneous requests, reset inhibition, recovery

In [ ]:
control.reset()
control.operate(data=5, serial=1, load=True, shift=True)
assert control.states() == (5, 1)
control.operate(data=12, serial=0, load=True, shift=True)
assert control.states() == (12, 2)
control.shift_bit(1)
assert control.states() == (12, 5)
control.load(9)
assert control.states() == (9, 5)

base = 15 | SERIAL | LOAD | SHIFT
control.write_command(base | RESET)
assert control.states() == (0, 0)
control.write_command(base)  # load/shift remain high during release
for _ in range(8):
    assert control.states() == (0, 0), "Commands executed unexpectedly after reset"
control.write_command(0)  # explicit rearm
control.operate(data=7, serial=1, load=True, shift=True)
assert control.states() == (7, 1)
control.reset()
assert control.states() == (0, 0)
print("Simultaneous commands, reset inhibition, and recovery: PASS")
print("ALL HARDWARE TESTS PASSED — only if run on the actual PYNQ-Z2.")

## 8. Validation notes

- Save this notebook **with its outputs** after running on the physical PYNQ-Z2. A blank notebook does not demonstrate hardware validation.
- Record the date, Vivado version, bitstream build/revision, **verified AXI GPIO base address**, board image version, and any failures in `docs/validation.md`.
- **No ACK** means these tests infer successful operation from observed states, not explicit per-command acknowledgement.
- MMIO tests do **not** replace GHDL RTL simulation or Vivado post-route timing/DRC review.
- If you modify or reprogram the bitstream, restart/re-run the notebook using the *new* address map and hardware configuration. If Vivado loading leaves AXI/PS uninitialized, configure the PS/clock/reset correctly before MMIO access.
